# Run the framework on Google Colab

Runs the code from GitHub `main` on a Colab GPU and saves results and caches to Google Drive
(`MyDrive/thesis_compression/`), so nothing is lost when the Colab session ends.

1. **Runtime > Change runtime type > GPU** (T4 is enough; pick High-RAM if you have it).
2. Run the cells top to bottom. Change `MODE` and the other settings in step 4 first if you want.
3. Rerunning later reuses the cached FP16 baseline and sensitivity profiles from Drive.

In [ ]:
# 1. Check the GPU and memory
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -h | head -2

In [ ]:
# 2. Mount Google Drive (results and caches are kept there)
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/thesis_compression"

In [ ]:
# 3. Get the latest code from GitHub (clone the first time, pull after that) and install it
BRANCH = "main"
REPO = "/content/sensitivity-driven-framework"
import os, sys
if os.path.isdir(REPO):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/hamuutpls/sensitivity-driven-framework {REPO}
!pip install -q -e "{REPO}[dev]"
!git -C {REPO} log --oneline -1
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")  # the running kernel does not see the editable install until a restart
sys.path.insert(0, REPO)  # so main.py can be imported below

In [ ]:
# 4. (optional) Run the test suite, a minute or two, no GPU or downloads needed
!python -m pytest -q

In [ ]:
# 5. Choose what to run. Every other setting keeps the value written in main.py.
import importlib, main
importlib.reload(main)  # pick up a fresh pull

main.MODE = "single"  # "single" | "sweep" | "compare_scores" | "prune_sweep"
main.SENSITIVITY_SCORE = "layer_removal"  # used by "single" and "prune_sweep"
main.OUTPUT_ROOT = f"{DRIVE_ROOT}/results"
main.CACHE_DIR = f"{DRIVE_ROOT}/cache"
main.RUN_ID = None  # None = new timestamped run; reuse a run's name to resume it

In [ ]:
# 6. Run (prints the paths of report.md, the Excel workbook and results.json on Drive)
main.main()